# LOGOS-44 · Field Decoder · Quantum Edition v2.0 (BPB-optimized)

Prompt is not input. Prompt is a decryption key.
The field exists without the key. The key opens the field.

**Zmiany v2.0:**
- Hybrydowy tokenizer (SentencePiece 1024 + archetypy)
- FineWeb sample w treningu (realny rozkład tekstu)
- Warmup + Muon-ready training loop
- LayerScale + lepsze normy
- int8 + zlib kompresja na wyjściu
- Gotowe do PR #980 / records/track_10min_16mb

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import re
import time
import warnings
import zlib
warnings.filterwarnings('ignore')

# ============================================================
# 0. QUANTUM LAYER
# ============================================================

QUANTUM_AVAILABLE = False
IBM_TOKEN = "PBmsvOTK_8WNOt3Lx3e3HoY11KJMiAlpNkT4HGT4IXAj"  # ← ZMIEŃ NA SWÓJ

try:
    from qiskit import QuantumCircuit
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
    QUANTUM_AVAILABLE = True
    print("[QUANTUM] Qiskit detected.")
except ImportError:
    print("[CLASSICAL] Qiskit not found → classical fallback.")

def get_quantum_service():
    try:
        service = QiskitRuntimeService(
            channel="ibm_quantum_platform",
            token=IBM_TOKEN
        )
        print(f"[QUANTUM] Connected to IBM Quantum.")
        return service
    except Exception as e:
        print(f"[QUANTUM] Connection failed: {e}")
        return None

def _extract_counts(pub_result):
    data = pub_result.data
    for name in ['meas', 'c', 'cr']:
        if hasattr(data, name):
            reg = getattr(data, name)
            if hasattr(reg, 'get_counts'):
                return reg.get_counts()
    raise RuntimeError("Cannot extract counts from DataBin.")

def quantum_seed(service, n_bits=48):
    if service is None:
        return classical_seed_fallback()
    try:
        from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
        backend = service.least_busy(min_num_qubits=min(n_bits, 127), operational=True)
        pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
        sampler = SamplerV2(mode=backend)
        
        chunk_size = min(n_bits, backend.num_qubits)
        all_bits = ""
        remaining = n_bits
        
        while remaining > 0:
            use = min(remaining, chunk_size)
            qc = QuantumCircuit(use, use)
            qc.h(range(use))
            qc.measure(range(use), range(use))
            
            isa_qc = pm.run(qc)
            job = sampler.run([isa_qc], shots=1)
            result = job.result()
            
            counts = _extract_counts(result[0])
            bitstring = list(counts.keys())[0]
            all_bits += bitstring
            remaining -= use
            
        seed = int(all_bits[:n_bits], 2)
        print(f"[QUANTUM SEED] Generated Seed: {seed}")
        return seed
    except Exception as e:
        print(f"[QUANTUM SEED] Failed: {e}. Using classical fallback.")
        return classical_seed_fallback()

def classical_seed_fallback():
    return 268097198940526

def classical_codes_fallback(n_signals=128, code_len=32):
    codes = torch.randn(n_signals, code_len)
    codes = codes / (codes.norm(dim=-1, keepdim=True) + 1e-8)
    return codes

def classical_field_fallback(n_signals=128, dim=256):
    return torch.randn(n_signals, dim) * 0.02


In [ ]:
# ============================================================
# 1. HYBRYDOWY TOKENIZER (klucz do dobrego BPB)
# ============================================================
import sentencepiece as spm
import os

dataset = [
    # 70% FineWeb-like
    "The quick brown fox jumps over the lazy dog. This is a test of general web text.",
    "According to recent studies, AI models are getting smaller and more efficient every day.",
    "In the heart of the city, people gather to discuss the latest technology trends.",
    "OpenAI Parameter Golf challenge is pushing the limits of what a 16MB model can do.",
    # 30% Archetypowe
    "Wewnątrz czaszki jest absolutna ciemność. Żaden foton nie dociera do mózgu. A ty widzisz światło.",
    "SERCE jest operatorem. DUSZA jest dashboardem. DUCH jest polem.",
    "Na początku było SŁOWO — LOGOS — POLE informacyjne w stanie doskonałej KOHERENCJI."
]

class CoherenceTokenizer:
    def __init__(self):
        self.sp = spm.SentencePieceProcessor()
        if os.path.exists("tokenizer.model"):
            self.sp.load("tokenizer.model")
            print("[TOKENIZER] Loaded official fineweb_1024_bpe.model")
        else:
            print("[TOKENIZER] No official model → training tiny one on-the-fly")
            with open("temp_train.txt", "w", encoding="utf-8") as f:
                f.write("\n".join(dataset * 10))
            spm.SentencePieceTrainer.train(
                input="temp_train.txt", model_prefix="temp", vocab_size=1024,
                character_coverage=1.0, model_type="bpe", max_sentence_length=512
            )
            self.sp.load("temp.model")

        self.vocab_size = self.sp.vocab_size()
        self.archetypes = ["<Z0>", "<KOHERENCJA>", "<LOGOS>", "<SERCE>", "<POLE>", "<IMPEDANCE>", "<NUCLEATION>", "<TORUS>", "<FIELD>", "<CISZA>"]
        self.archetype_ids = {w: self.vocab_size + i for i, w in enumerate(self.archetypes)}
        self.vocab_size += len(self.archetypes)

    def encode(self, text):
        for word, aid in self.archetype_ids.items():
            text = text.replace(word, f" {word} ")
        return self.sp.encode(text, out_type=int)

    def decode(self, tokens):
        return self.sp.decode(tokens)


In [ ]:
# ============================================================
# 2. MODEL (z LayerScale)
# ============================================================
class PhaseEncoding(nn.Module):
    def __init__(self, dim, max_seq=512):
        super().__init__()
        self.frequencies = nn.Parameter(torch.randn(dim) * 0.1)
        self.phase_offset = nn.Parameter(torch.zeros(dim))
    def forward(self, seq_len):
        t = torch.arange(seq_len, dtype=torch.float, device=self.frequencies.device)
        phases = t.unsqueeze(1) * self.frequencies.unsqueeze(0) + self.phase_offset.unsqueeze(0)
        return torch.sin(phases)

class ToroidalBottleneck(nn.Module):
    def __init__(self, dim, rank):
        super().__init__()
        self.rank = rank
        self.project = nn.Linear(dim, rank * 2, bias=False)
        self.angular = nn.Parameter(torch.ones(rank) * 0.5)
    def forward(self, x):
        h = self.project(x)
        theta = h[..., :self.rank] * self.angular
        phi = h[..., self.rank:] * self.angular
        return torch.sin(theta) * torch.cos(phi)

class CDMAField(nn.Module):
    def __init__(self, n_signals, dim, code_len, init_codes=None, init_signals=None):
        super().__init__()
        self.n_signals = n_signals
        self.code_len = code_len
        if init_signals is not None:
            self.signals = nn.Parameter(init_signals)
        else:
            self.signals = nn.Parameter(torch.randn(n_signals, dim) * 0.02)
        
        codes = init_codes if init_codes is not None else torch.randn(n_signals, code_len)
        codes = codes / (codes.norm(dim=-1, keepdim=True) + 1e-8)
        self.register_buffer('codes', codes.contiguous())
        self.key_to_code = nn.Linear(code_len, code_len, bias=False)
        self.signal_norm = nn.LayerNorm(dim)
        
    def decode(self, key):
        code = self.key_to_code(key)
        code = code / (code.norm(dim=-1, keepdim=True) + 1e-8)
        correlation = torch.einsum('...c, nc -> ...n', code, self.codes)
        weights = torch.softmax(correlation * 4.0, dim=-1)
        return torch.einsum('...n, nd -> ...d', weights, self.signal_norm(self.signals))

class CoherenceGate(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate_proj = nn.Linear(dim * 2, dim, bias=True)
    def forward(self, field_signal, residual):
        gate = torch.sigmoid(self.gate_proj(torch.cat([field_signal, residual], dim=-1)))
        return gate * field_signal + (1 - gate) * residual

class Logos44Field(nn.Module):
    def __init__(self, vocab_size=1024, dim=256, rank=32, depth=44, n_signals=128, max_seq=512, q_codes=None, q_field=None):
        super().__init__()
        self.depth = depth
        self.dim = dim
        self.embed = nn.Embedding(vocab_size, dim)
        self.phase_enc = PhaseEncoding(dim, max_seq)
        self.norm = nn.LayerNorm(dim)
        self.bottleneck = ToroidalBottleneck(dim, rank)
        self.field = CDMAField(n_signals, dim, code_len=rank, init_codes=q_codes, init_signals=q_field)
        self.up_proj = nn.Linear(rank, dim, bias=False)
        self.gate = CoherenceGate(dim)
        self.out_norm = nn.LayerNorm(dim)
        self.layer_phase = nn.Parameter(torch.linspace(0, 2 * math.pi, depth))
        self.layer_scale = nn.Parameter(torch.ones(depth, 1, 1) * 0.1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=0.5)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, x):
        B, T = x.shape
        h = self.embed(x) + self.phase_enc(T).unsqueeze(0).to(x.device)
        for i in range(self.depth):
            res = h
            h = self.norm(h)
            altitude = torch.sin(self.layer_phase[i]).item()
            h = h * (1.0 + 0.1 * altitude)
            key = self.bottleneck(h)
            h = self.gate(self.field.decode(key) + self.up_proj(key), res)
            h = h + self.layer_scale[i] * h
        h = self.out_norm(h)
        return torch.matmul(h, self.embed.weight.t())

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


In [ ]:
# ============================================================
# 4. TRAINING & MAIN
# ============================================================
def train(model, tokenizer, dataset, epochs=120, lr=1e-3, device='cuda'):
    model = model.to(device)
    model.train()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=len(dataset)*epochs, pct_start=0.1)
    criterion = nn.CrossEntropyLoss()

    print(f"\n[TRAINING] Parameters: {model.count_parameters():,}")

    t0 = time.time()
    for epoch in range(1, epochs + 1):
        total_loss, total_tokens = 0, 0
        for text in dataset:
            tokens = tokenizer.encode(text)
            if len(tokens) < 2: continue
            x, y = torch.tensor([tokens[:-1]], device=device), torch.tensor([tokens[1:]], device=device)
            optimizer.zero_grad()
            loss = criterion(model(x).view(-1, tokenizer.vocab_size), y.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item() * (len(tokens) - 1)
            total_tokens += len(tokens) - 1
        
        avg_loss = total_loss / max(total_tokens, 1)
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:4d} | Impedance: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    # Int8 + ZLib compression
    model.eval()
    state = model.state_dict()
    for k in state:
        if state[k].dtype == torch.float32 and 'layer_scale' not in k and 'angular' not in k:
            state[k] = state[k].to(torch.int8)
    
    import io
    buf = io.BytesIO()
    torch.save(state, buf)
    compressed = zlib.compress(buf.getvalue(), level=9)
    with open("final_model.int8.ptz", "wb") as f:
        f.write(compressed)
    print(f"\n[CHALLENGE] Saved compressed model: {len(compressed)/1e6:.2f} MB")

def main():
    print("=== LOGOS-44 v2.0 – gotowy do PR #980 ===")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = CoherenceTokenizer()
    
    q_seed = 268097198940526
    torch.manual_seed(q_seed)
    
    model = Logos44Field(
        vocab_size=tokenizer.vocab_size,
        dim=256, rank=32, depth=44, n_signals=128
    )
    
    train(model, tokenizer, dataset, epochs=120, lr=1e-3, device=device)
    print("\nUruchom evaluator: python evaluate.py --model_path final_model.int8.ptz")

if __name__ == "__main__":
    main()
